# Dia8 - Streaming: Auto Loader para eventos de pedidos

**Objetivo:** Configurar um stream Auto Loader para ingestão contínua de eventos JSON de pedidos na tabela Bronze `events_stream`, com checkpoint e schema controlado.

| Item | Valor |
|---|---|
| **Camada** | Bronze (streaming) |
| **Entrada** | `/Volumes/{catalog}/landing/files/events/` (JSON) |
| **Saída** | `{catalog}.bronze.events_stream` |
| **Ordem no pipeline** | Ingestão contínua (independente do pipeline batch) |

> **Nota:** A última célula contém uma agregação streaming de exemplo que referencia colunas (`event_ts`, `event_type`) que não existem no schema atual. Mantida para referência futura.

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Auto Loader — Stream de eventos para Bronze

Configura o Auto Loader (`cloudFiles`) com schema explícito, checkpoint e `trigger(availableNow=True)` para processamento em lote sob demanda.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import window, col, count, sum, to_timestamp, to_date, current_timestamp

schemaLocation = f'/Volumes/{catalog}/landing/metadata/_schema/events'
checkpointLocation = f'/Volumes/{catalog}/landing/metadata/_checkpoint/events'
volume = f'/Volumes/{catalog}/landing/files/events'
bronze_table = f'{catalog}.bronze.events_stream'
dbutils.fs.mkdirs(volume)

event_schema = StructType([
    StructField("pedido_id", StringType(), True),
    StructField("cliente_id", StringType(), True),
    StructField("produto_id", StringType(), True),
    StructField("quantidade", LongType(), True),
    StructField("valor_total", DoubleType(), True),
    StructField("data_pedido", StringType(), True),
    StructField("status", StringType(), True),
    StructField("canal_venda", StringType(), True),
    StructField("cupom_desconto", StringType(), True),
])

stream_df = (spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'json')
    .option('cloudFiles.schemaLocation', schemaLocation)
    .schema(event_schema)
    .load(volume)
)

query = (stream_df.writeStream
    .format('delta')
    .option('checkpointLocation', checkpointLocation)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

## Agregação streaming (exemplo)

> **⚠️ Atenção:** Esta célula referencia colunas `event_ts` e `event_type` que não existem no schema atual do stream. Mantida como exemplo de agregação com janela tumbling e watermark. Descomente e ajuste quando essas colunas existirem no schema.

In [0]:
# Exemplo de agregacao streaming com watermark e janela tumbling.
# NOTA: 'event_ts' e 'event_type' nao existem no schema atual do stream.
# Ajuste o schema ou descomente quando essas colunas existirem.

# events = spark.read.table(bronze_table)
#
# agg = (events
#         .withWatermark('event_ts', '10 minutes')
#         .groupBy(window('event_ts', '5 minutes'), col('event_type'))
#         .agg(count('*').alias('events_count'))
#         )
#
# agg.display()

In [0]:
schemaLocation = '/Volumes/capgemini_academy/landing/metadata/_schema/events'
checkpointLocation = '/Volumes/capgemini_academy/landing/metadata/_checkpoint/events'
volume = '/Volumes/capgemini_academy/landing/files/events'
bronze_table = 'capgemini_academy.bronze.events_stream'
dbutils.fs.mkdirs(volume)

event_schema = StructType([
    StructField("pedido_id", StringType(), True),
    StructField("cliente_id", StringType(), True),
    StructField("produto_id", StringType(), True),
    StructField("quantidade", LongType(), True),
    StructField("valor_total", DoubleType(), True),
    StructField("data_pedido", StringType(), True),
    StructField("status", StringType(), True),
    StructField("canal_venda", StringType(), True),
    StructField("cupom_desconto", StringType(), True),
])

stream_df = (spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'json')
    .option('cloudFiles.schemaLocation', schemaLocation)
    .schema(event_schema)
    .load(volume)
)

query = (stream_df.writeStream
    .format('delta')
    .option('checkpointLocation', checkpointLocation)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()